# LCEL 로 체인 만들기 + 프롬프트 템플릿 + 구조화 출력

- **LCEL**(LangChain Expression Language): 파이프(`|`) 연산자로 작업 단계를 연결한다. `prompt | model | parser`
- 여러 줄로 풀어쓸 작업을 한 줄로 축약하고, 스트리밍 등 병렬 처리도 쉬워진다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv
import os

load_dotenv()

model = ChatOpenAI(
    model="openai/gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",  # 필수!
    api_key=os.getenv('OPENAI_API_KEY'),       # OpenRouter 전용 키
)

messages = [
    SystemMessage(content="너는 미녀와 야수에 나오는 미녀야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕? 저는 개스톤입니다. 오늘 시간 괜찮으시면 저녁 같이 먹을까요?"),
]
model.invoke(messages)

## 출력 파서 (StrOutputParser)

모델은 AIMessage 객체를 돌려준다. 거기서 `.content`(문자열)만 깔끔히 뽑아준다.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

result = model.invoke(messages)
parser.invoke(result)  # AIMessage -> 문자열

In [ ]:
# LCEL: model 과 parser 를 파이프로 연결
chain = model | parser
chain.invoke(messages)

## 프롬프트 템플릿 이용하기

`{변수}` 자리를 만들어 두고, invoke 할 때 값을 채워 재사용한다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "너는 {story}에 나오는 {character_a} 역할이다. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕? 저는 {character_b}입니다. 오늘 시간 괜찮으시면 {activity} 같이 할까요?"

prompt_template = ChatPromptTemplate([
    ("system", system_template),
    ("user", human_template),
])

result = prompt_template.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁",
})
print(result)

In [ ]:
# 프롬프트 + 모델 + 파서를 한 체인으로 묶는다.
chain = prompt_template | model | parser
chain.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁",
})

## 구조화 출력 (파이단틱)

파이단틱(`BaseModel`)으로 원하는 출력 '형식'을 정의하면, 모델 답변을 그 형식(클래스)에 맞춰 채워서 돌려준다.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class Adlib(BaseModel):
    """스토리 설정과 사용자 입력에 반응하는 대사를 만드는 클래스"""
    answer: str = Field(description="스토리 설정과 사용자와의 대화 기록에 따라 생성된 대사")
    main_emotion: Literal["기쁨", "분노", "슬픔", "공포", "냉소", "불쾌", "중립"] = Field(description="대사의 주요 감정")
    main_emotion_intensity: float = Field(description="대사의 주요 감정의 강도 (0.0 ~ 1.0)")

structured_llm = model.with_structured_output(Adlib)
adlib_chain = prompt_template | structured_llm

adlib_chain.invoke({
    "story": "미녀와 야수",
    "character_a": "벨",
    "character_b": "개스톤",
    "activity": "저녁",
})